## We will follow the medallion structure of a data pipeline, referencing the below image:

![alt text](https://imgs.search.brave.com/VlB9xn1-mwTi41UwDSwN6lZMdcd6r4S86akLVXeIW-A/rs:fit:860:0:0:0/g:ce/aHR0cHM6Ly9jbGlj/a2hvdXNlLmNvbS9f/bmV4dC9pbWFnZT91/cmw9L3VwbG9hZHMv/ZGlhZ3JhbV8xX21l/ZGFsbGlvbl82NTc1/ZDhkNTBlLnBuZyZ3/PTIwNDgmcT03NQ "Medallion Pipeline")

#### Objectives:
1. Create Silver layer tables in new folder called "Tables_silver"
2. Silver layer schema:

![silver layer tables](silver%20layer%20schema.png)

In [127]:
import pandas as pd
import hashlib
import json
import os
import glob
from faker import Faker

#### Objectives:
0. De-duplicate the bronze table
0. type coercion on dtypes: str, float64, int
2. Populate Employees_silver
3. Populate Currencies_silver
4. Populate Customers_silver
5. Populate Transactions_silver
6. Store all to Tables_silver directory

In [128]:
# read all bronze data into preliminary DataFrame
path = 'C:/Users/miche/OneDrive/Documents/FCEDataApp/Masked Data_bronze'

# search nested folders for CSV files
bronze_files = glob.glob(path + "/**/*.csv", recursive=True)

bronze_df = pd.concat([pd.read_csv(file, parse_dates=['Date and Time Entered', 'Date']) for file in bronze_files])

C:\Users\miche\AppData\Local\Temp\ipykernel_10308\3710216730.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  bronze_df = pd.concat([pd.read_csv(file, parse_dates=['Date and Time Entered', 'Date']) for file in bronze_files])
C:\Users\miche\AppData\Local\Temp\ipykernel_10308\3710216730.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  bronze_df = pd.concat([pd.read_csv(file, parse_dates=['Date and Time Entered', 'Date']) for file in bronze_files])
C:\Users\miche\AppData\Local\Temp\ipykernel_10308\3710216730.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  bronze_df =

In [129]:
#check dtypes as loaded
bronze_df.dtypes

uid                          str
#                          int64
Transaction ID             int64
Reservation ID           float64
User/Preparer                str
                       ...      
Live Rate Feed           float64
ID Number                 object
ID Type                   object
Date              datetime64[us]
Location                     str
Length: 66, dtype: object

In [130]:
# drop duplicate rows if any
bronze_df = bronze_df.drop_duplicates()

In [131]:
# list columns for type coercion
bronze_df.columns

Index(['uid', '#', 'Transaction ID', 'Reservation ID', 'User/Preparer', 'Type',
       'Date and Time Entered', 'Current Status', 'Currency Code',
       'Product Type', 'Amount', 'Rate', 'USD Value', 'Client Total Fees',
       'Transaction Total', 'Account Holder', 'Citizen/Resident',
       'Client Name', 'Client Fee Override Reason',
       'Client Fee Override Approved By', 'Client Rate Override Approved By',
       'Client Rate Override Reason', 'Client Type', 'Company Name',
       'Company Type', 'Country of Issuance', 'Date of Birth', 'E-mail',
       'Approved By', 'Posted Spot Rate', 'Posted Spot Value', 'Posted Spread',
       'Fee', 'Fee Received', 'Entered Spot Rate', 'Entered Spot Value',
       'Entered Spread', 'Denominations', 'Currency Name',
       'Client Converted Fees', 'Base Customer Rate', 'Base Customer Value',
       'Basic Denomination Choice', 'Action', 'Client Converted Base Value',
       'Type of Business', 'Total Posted Spread', 'Total Entered Spread',


In [132]:
# set mapping of dtypes for each column
convert_map = {'uid' : str, '#': int, 'Transaction ID': str, 'Reservation ID': str, 'User/Preparer': str, 'Type': str,
        'Current Status': str, 'Currency Code': str,
       'Product Type': str, 'Amount': float, 'Rate': float, 'USD Value': float, 'Client Total Fees': float,
       'Transaction Total': float, 'Account Holder': str, 'Citizen/Resident': str,
       'Client Name': str, 'Client Fee Override Reason': str,
       'Client Fee Override Approved By': str, 'Client Rate Override Approved By': str,
       'Client Rate Override Reason': str, 'Client Type': str, 'Company Name': str,
       'Company Type': str, 'Country of Issuance': str, 'Date of Birth': str, 'E-mail': str,
       'Approved By': str, 'Posted Spot Rate': float, 'Posted Spot Value': float, 'Posted Spread': float,
       'Fee': float, 'Fee Received': float, 'Entered Spot Rate': float, 'Entered Spot Value': float,
       'Entered Spread': float, 'Denominations': str, 'Currency Name': str,
       'Client Converted Fees': float, 'Base Customer Rate': float, 'Base Customer Value': float,
       'Basic Denomination Choice': str, 'Action': str, 'Client Converted Base Value': float,
       'Type of Business': str, 'Total Posted Spread': float, 'Total Entered Spread': float,
       'State/Province of Issuance': str, 'State/Province': str, 'Source of Funds': str,
       'Source': str, 'Signature is Required': str, 'Return Reason': str, 'Profit': float,
       'Rate Types': str, 'Phone': str, 'Payment Types': str, 'Occupation': str, 'Notes': str,
       'Method of Payment': str, 'Live Rate Feed': str, 'ID Number': str, 'ID Type': str, 
       'Location': str}

In [133]:
# apply dtype mapping to dataframe
bronze_df = bronze_df.astype(convert_map)

In [134]:
# check results of type coercion
bronze_df.dtypes

uid                          str
#                          int64
Transaction ID               str
Reservation ID               str
User/Preparer                str
                       ...      
Live Rate Feed               str
ID Number                    str
ID Type                      str
Date              datetime64[us]
Location                     str
Length: 66, dtype: object

#### Objectives:
0. ~~De-duplicate the bronze table~~
0. ~~type coercion on dtypes: str, float64, int~~
2. Populate Employees_silver
3. Populate Currencies_silver
4. Populate Customers_silver
5. Populate Transactions_silver
6. Store all to Tables_silver directory

In [135]:
# create employee table as unique employee names
Employees_silver = pd.DataFrame(bronze_df['User/Preparer'].drop_duplicates())
Employees_silver.rename(columns={'User/Preparer': 'Employee Name'}, inplace=True)

In [136]:
# create function to assign employee IDs

def assign_employee_id(employee_name):
    x = "".join([word[0] for word in employee_name.split()]).upper() + str(len(employee_name)) #DO NOT USE LEN IN FINAL, USE FAKER LIBRARY TO GENERATE NUMERICAL PART  
    return x

In [137]:
# assign employee IDs to each employee name
Employees_silver['Employee_ID'] = Employees_silver['Employee Name'].apply(lambda x: assign_employee_id(x))

In [138]:
Employees_silver.head()

,Employee Name,Employee_ID
0,Maureen Bullock,MB15
0,Christopher Morales,CM19
0,John Garcia,JG11
0,Paul Morgan,PM11
0,Teresa Russo,TR12


#### Objectives:
0. ~~De-duplicate the bronze table~~
0. ~~type coercion on dtypes: str, float64, int~~
2. ~~Populate Employees_silver~~
3. Populate Currencies_silver
4. Populate Customers_silver
5. Populate Transactions_silver
6. Store all to Tables_silver directory

#### NOTE: Employee table exists in source DB but will be generated from already masked data for security reasons
#### This is just an example pipeline with a focus on masking data and ETL of medallion framework

In [139]:
Currencies_silver = pd.DataFrame(bronze_df[['Currency Code', 'Product Type', 'Currency Name']].drop_duplicates())

In [140]:
Currencies_silver

,Currency Code,Product Type,Currency Name
0,EUR,CASH,European Central Bank Euro
2,NZD,CASH,New Zealand Dollar
0,JPY,CASH,Japan Yen
1,CRC,CASH,Costa Rica Colones
3,CAD,CASH,Canada Dollar
...,...,...,...
5,BZD,CASH,Belize Dollar
3,KHR,CASH,Cambodia Riel
1,UAH,CASH,Ukraine Hryvnia
0,JOD,CASH,Jordan Dinar


#### Objectives:
0. ~~De-duplicate the bronze table~~
0. ~~type coercion on dtypes: str, float64, int~~
2. ~~Populate Employees_silver~~
3. ~~Populate Currencies_silver~~
4. Populate Customers_silver
5. Populate Transactions_silver
6. Store all to Tables_silver directory

In [141]:
Customers_silver = pd.DataFrame(bronze_df[['Client Name', 'Client Type', 'Account Holder', 'Citizen/Resident', 
                                           'Country of Issuance', 'State/Province of Issuance', 'State/Province', 'Date of Birth', 
                                           'E-mail', 'Phone', 'ID Number', 'ID Type','Occupation']].drop_duplicates())


In [142]:
# create function to assign customer IDs

def assign_customer_id(customer_name):
    x = "".join([word[0] for word in str(customer_name).split()]).upper() + str(len(str(customer_name))) #DO NOT USE LEN IN FINAL, USE FAKER LIBRARY TO GENERATE NUMERICAL PART  
    return x

In [143]:
# assign customer IDs to each customer name
Customers_silver['Customer_ID'] = Customers_silver['Client Name'].apply(lambda x: assign_customer_id(x))  

In [144]:
Customers_silver

,Client Name,Client Type,Account Holder,Citizen/Resident,Country of Issuance,State/Province of Issuance,State/Province,Date of Birth,E-mail,Phone,ID Number,ID Type,Occupation,Customer_ID
0,Jennifer Craig,Personal,No,No,Italy,NaN,NaN,11/27/2000,NaN,NaN,4107465c9f40b22a28cc069c896fbb82e8806271a5692c...,Foreign Passport,NaN,JC14
1,Martha Rubio,Personal,No,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,MR12
2,Steven Montgomery,Personal,No,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,SM17
3,Joshua Keller,Personal,No,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,JK13
0,Brianna Price,Personal,No,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,BP13
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
0,Scott Dixon,Personal,No,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,SD11
1,Kathleen Taylor,Personal,No,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,KT15
0,Erica Cardenas,Personal,No,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,EC14
3,Karen Davis,Personal,No,Yes,United States,Florida,NaN,11/03/1941,NaN,NaN,151e675e0bf3dba73659503dab322c8329d150db865dff...,Driver License,NaN,KD11


#### Objectives:
0. ~~De-duplicate the bronze table~~
0. ~~type coercion on dtypes: str, float64, int~~
2. ~~Populate Employees_silver~~
3. ~~Populate Currencies_silver~~
4. ~~Populate Customers_silver~~
5. Populate Transactions_silver
6. Store all to Tables_silver directory

In [145]:
Txns_silver = pd.DataFrame(bronze_df[['uid','Date','Date and Time Entered','Location','Type','Currency Code','Currency Name','Amount', 'Rate', 
                                      'USD Value','Transaction Total', 'Fee', 'Fee Received', 'Client Total Fees', 'Client Converted Fees', 
                                      'Entered Spot Rate', 'Posted Spot Rate','Base Customer Rate','Live Rate Feed','Rate Types', 
                                      'Entered Spot Value','Posted Spot Value','Client Converted Base Value','Base Customer Value', 
                                      'Posted Spread', 'Entered Spread', 'Total Posted Spread', 'Total Entered Spread', 
                                      'Approved By', 'Client Rate Override Approved By', 'Client Rate Override Reason', 
                                      'Client Fee Override Approved By', 'Client Fee Override Reason', 
                                      'Profit', 
                                      'Basic Denomination Choice', 'Denominations', 
                                      'Type of Business','Payment Types', 'Method of Payment','Source','Source of Funds', 
                                      'Signature is Required', 'Action', 'Return Reason', 
                                      'Transaction ID', 'Reservation ID',
                                      'Notes', 'Current Status', 'User/Preparer', 'Client Name']])

In [146]:
# add in employees to transactions table by merging on name fields
Txns_silver = Txns_silver.merge(Employees_silver, left_on='User/Preparer', right_on='Employee Name', how='left')

In [147]:
#add in customers to transactions table by merging on name fields
Txns_silver = Txns_silver.merge(Customers_silver[['Client Name', 'Customer_ID']], left_on='Client Name', right_on='Client Name', how='left')

In [148]:
Txns_silver.drop(columns=['User/Preparer', 'Client Name', 'Employee Name'], inplace=True)
Txns_silver.head()

,uid,Date,Date and Time Entered,Location,Type,Currency Code,Currency Name,Amount,Rate,USD Value,...,Source of Funds,Signature is Required,Action,Return Reason,Transaction ID,Reservation ID,Notes,Current Status,Employee_ID,Customer_ID
0,26791758_EUR,2026-01-02,2026-01-02 15:59:00,Karlaport,Purchases,EUR,European Central Bank Euro,1900.0,1.110330,2110.0,...,NaN,NaN,Entered,NaN,26791758,NaN,NaN,Inventory Posted,MB15,JC14
1,26791780_EUR,2026-01-02,2026-01-02 16:02:00,Karlaport,Purchases,EUR,European Central Bank Euro,500.0,1.110000,555.0,...,NaN,NaN,Entered,NaN,26791780,NaN,NaN,Inventory Posted,MB15,MR12
2,26791818_NZD,2026-01-02,2026-01-02 16:06:00,Karlaport,Purchases,NZD,New Zealand Dollar,1000.0,0.520000,520.0,...,NaN,NaN,Entered,NaN,26791818,NaN,NaN,Inventory Posted,MB15,SM17
3,26792825_EUR,2026-01-02,2026-01-02 18:27:00,Karlaport,Purchases,EUR,European Central Bank Euro,200.0,1.100000,220.0,...,NaN,NaN,Entered,NaN,26792825,NaN,NaN,Inventory Posted,MB15,JK13
4,26789260_JPY,2026-01-02,01/02/2026 11:20 am,Morganberg,Sales,JPY,Japan Yen,29000.0,0.006887,200.0,...,NaN,NaN,Entered,NaN,26789260,NaN,NaN,Inventory Posted,CM19,BP13


In [149]:
# Reorder columns to place IDs at the front
Txns_silver = Txns_silver[['uid', 'Date', 'Date and Time Entered', 'Location', 'Employee_ID', 'Customer_ID', 'Type', 
                           'Currency Code', 'Currency Name', 'Amount', 'Rate', 'USD Value', 'Transaction Total', 
                           'Fee', 'Fee Received', 'Client Total Fees', 'Client Converted Fees', 
                           'Entered Spot Rate', 'Posted Spot Rate','Base Customer Rate','Live Rate Feed','Rate Types',
                           'Entered Spot Value','Posted Spot Value','Client Converted Base Value','Base Customer Value',
                           'Posted Spread', 'Entered Spread', 'Total Posted Spread', 'Total Entered Spread',
                           'Approved By', 'Client Rate Override Approved By', 'Client Rate Override Reason',
                           'Client Fee Override Approved By', 'Client Fee Override Reason',
                           'Profit',
                           'Basic Denomination Choice', 'Denominations',
                           'Type of Business','Payment Types', 'Method of Payment','Source','Source of Funds',
                           'Signature is Required', 'Action', 'Return Reason','Transaction ID', 'Reservation ID',
                           'Notes', 'Current Status']]

In [150]:
Txns_silver.head()

,uid,Date,Date and Time Entered,Location,Employee_ID,Customer_ID,Type,Currency Code,Currency Name,Amount,...,Method of Payment,Source,Source of Funds,Signature is Required,Action,Return Reason,Transaction ID,Reservation ID,Notes,Current Status
0,26791758_EUR,2026-01-02,2026-01-02 15:59:00,Karlaport,MB15,JC14,Purchases,EUR,European Central Bank Euro,1900.0,...,NaN,UI,NaN,NaN,Entered,NaN,26791758,NaN,NaN,Inventory Posted
1,26791780_EUR,2026-01-02,2026-01-02 16:02:00,Karlaport,MB15,MR12,Purchases,EUR,European Central Bank Euro,500.0,...,NaN,UI,NaN,NaN,Entered,NaN,26791780,NaN,NaN,Inventory Posted
2,26791818_NZD,2026-01-02,2026-01-02 16:06:00,Karlaport,MB15,SM17,Purchases,NZD,New Zealand Dollar,1000.0,...,NaN,UI,NaN,NaN,Entered,NaN,26791818,NaN,NaN,Inventory Posted
3,26792825_EUR,2026-01-02,2026-01-02 18:27:00,Karlaport,MB15,JK13,Purchases,EUR,European Central Bank Euro,200.0,...,NaN,UI,NaN,NaN,Entered,NaN,26792825,NaN,NaN,Inventory Posted
4,26789260_JPY,2026-01-02,01/02/2026 11:20 am,Morganberg,CM19,BP13,Sales,JPY,Japan Yen,29000.0,...,NaN,UI,NaN,NaN,Entered,NaN,26789260,NaN,NaN,Inventory Posted


#### Objectives:
0. ~~De-duplicate the bronze table~~
0. ~~type coercion on dtypes: str, float64, int~~
2. ~~Populate Employees_silver~~
3. ~~Populate Currencies_silver~~
4. ~~Populate Customers_silver~~
5. ~~Populate Transactions_silver~~
6. Store all to Tables_silver directory

In [151]:
#write employee table to CSV
Employees_silver.to_csv('C:/Users/miche/OneDrive/Documents/FCEDataApp/Tables_silver/Employees_silver.csv', index=True)

#write customer table to CSV
Customers_silver.to_csv('C:/Users/miche/OneDrive/Documents/FCEDataApp/Tables_silver/Customers_silver.csv', index=True)

#write transactions table to CSV
Txns_silver.to_csv('C:/Users/miche/OneDrive/Documents/FCEDataApp/Tables_silver/Txns_silver.csv', index=True)